# BirdCLEF+ 2025 — Pre-computed whole-clip mel spectrograms (Member 1)

The baseline spent ~17 of its ~32 minutes turning audio into spectrograms, and it only
used the **first 5 s** of each clip. This notebook does the conversion **once** for the
**whole clip** (capped at `max_sec`), so every later training run can:

* skip the 17-minute pre-processing step, and
* cut a **random** 5 s window from anywhere in the clip (the random-crop ablation).

It also saves spectrograms of some unlabelled `train_soundscapes`, used as background noise
in the noise + mixup ablation.

**Mel settings are identical to the baseline** (n_fft 2048, hop 512, 128 mels, 50–14,000 Hz,
`power_to_db(ref=max, top_db=80)` scaled to [0, 1]).

### Outputs
| File | Contents |
|---|---|
| `train_mels.npy` | uint8 array `(total_frames, 128)`: all clips stacked along time |
| `train_index.csv` | one row per clip: `filename`, `primary_label`, `secondary_labels`, `start`, `n_frames` |
| `noise_mels.npy` | uint8 array `(n_noise, frames_60s, 128)` from unlabelled soundscapes |
| `noise_index.csv` | which soundscape each noise row came from |
| `mels_cfg.json` | settings, and how to decode the uint8 values |

### How to read them in a training notebook
```python
X   = np.load(".../train_mels.npy", mmap_mode="r")      # or without mmap to load into RAM
idx = pd.read_csv(".../train_index.csv")
r   = idx.iloc[i]
db  = X[r.start : r.start + r.n_frames].T / 255.0 * 120 - 120   # (128, n_frames), dB re clip max

# A 5 s window, normalised exactly like inference does for each test window:
w = db[:, off : off + 313]
w = (np.maximum(w - w.max(), -80) + 80) / 80                   # [0, 1], same as the baseline
```
**Why 120 dB is stored, not 80.** Each clip is normalised by the loudest point of the
**whole** clip, but inference normalises each 5 s window on its own and keeps the 80 dB below
*that window's* peak. If only 80 dB were stored, a crop from a quiet part would lose its faint
background (it would be flattened to the floor). Storing 120 dB keeps that detail, so the
re-normalised crop matches inference exactly unless the crop is more than 40 dB quieter than
the clip's loudest moment (those crops are close to silent anyway).

In [ ]:
import os, glob, json, time
from pathlib import Path
import multiprocessing as mp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

CFG = dict(
    sr=32_000, n_fft=2048, hop_length=512, n_mels=128, fmin=50, fmax=14_000, top_db=80,
    store_db=120,      # dB range kept on disk (more than top_db, see above)
    max_sec=30,        # cap per training clip (median clip is 20.5 s)
    min_sec=5,         # shorter clips are zero-padded to 5 s so a full window exists
    n_noise=1000,      # unlabelled soundscapes kept for background noise
    noise_sec=60,
    seed=42,
)

def find_data_dir():
    if os.environ.get("MELS_DATA_DIR"):             # local smoke test
        return Path(os.environ["MELS_DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

DATA = find_data_dir()
OUT = Path(os.environ.get("MELS_OUT_DIR", "."))
OUT.mkdir(parents=True, exist_ok=True)
LIMIT = int(os.environ.get("MELS_LIMIT", "0"))     # >0 only for a quick local test
print("Data:", DATA, "| out:", OUT.resolve(), "| CPUs:", os.cpu_count(), "| limit:", LIMIT or "none")

## Mel function
Same as the baseline, but stored as **uint8** (0–255) instead of float16 to halve the size.
Values cover the 120 dB below the clip's loudest point, so one uint8 step is
120 / 255 ≈ 0.5 dB, far below what matters for training.

In [ ]:
def to_mel_u8(y):
    m = librosa.feature.melspectrogram(
        y=y, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    db = librosa.power_to_db(m, ref=np.max, top_db=CFG["store_db"])  # [-120, 0]
    x = (db + CFG["store_db"]) / CFG["store_db"]                      # [0, 1]
    return np.round(x * 255).astype(np.uint8).T                       # (frames, n_mels)

def n_samples_for(n_file):
    """How many samples we keep from a file with n_file samples."""
    return max(min(n_file, CFG["max_sec"] * CFG["sr"]), CFG["min_sec"] * CFG["sr"])

def n_frames_for(n_samples):
    return 1 + n_samples // CFG["hop_length"]     # librosa default (center=True)

def read_audio(path, n_keep):
    y, sr = sf.read(path, frames=n_keep, dtype="float32", always_2d=True)
    assert sr == CFG["sr"], f"{path}: sample rate {sr}"
    y = y.mean(axis=1)
    if len(y) < n_keep:
        y = np.pad(y, (0, n_keep - len(y)))
    return y

def train_clip_mel(args):
    fname, n_keep = args
    return to_mel_u8(read_audio(DATA / "train_audio" / fname, n_keep))

## Training clips
First read every file's length (fast, header only) so we know where each clip goes in one big
array. Then convert in parallel and write each result straight into that array on disk, so
memory use stays low.

In [ ]:
train = pd.read_csv(DATA / "train.csv")
if LIMIT:
    train = train.sample(LIMIT, random_state=CFG["seed"]).reset_index(drop=True)

t0 = time.time()
n_file = np.array([sf.info(DATA / "train_audio" / f).frames for f in train["filename"]])
n_keep = np.array([n_samples_for(n) for n in n_file])
n_frames = np.array([n_frames_for(n) for n in n_keep])
start = np.concatenate([[0], np.cumsum(n_frames)[:-1]])
total = int(n_frames.sum())
print(f"{len(train)} clips | header scan {time.time() - t0:.0f}s")
print(f"kept audio: {n_keep.sum() / CFG['sr'] / 3600:.1f} h | "
      f"clips cut at {CFG['max_sec']} s: {(n_file > CFG['max_sec'] * CFG['sr']).mean():.1%} | "
      f"padded to {CFG['min_sec']} s: {(n_file < CFG['min_sec'] * CFG['sr']).mean():.1%}")
print(f"output size: {total * CFG['n_mels'] / 1e9:.2f} GB ({total:,} frames)")

In [ ]:
X = np.lib.format.open_memmap(OUT / "train_mels.npy", mode="w+", dtype=np.uint8,
                              shape=(total, CFG["n_mels"]))
t0 = time.time()
jobs = list(zip(train["filename"], n_keep))
with mp.get_context("fork").Pool(os.cpu_count()) as pool:
    for i, mel in enumerate(pool.imap(train_clip_mel, jobs, chunksize=16)):
        assert mel.shape[0] == n_frames[i], (train["filename"][i], mel.shape, n_frames[i])
        X[start[i] : start[i] + n_frames[i]] = mel
        if (i + 1) % 2000 == 0 or i + 1 == len(jobs):
            el = time.time() - t0
            print(f"  {i + 1:6d}/{len(jobs)} clips | {el / 60:5.1f} min | "
                  f"~{el / (i + 1) * (len(jobs) - i - 1) / 60:.0f} min left", flush=True)
X.flush(); del X

index = train[["filename", "primary_label", "secondary_labels"]].copy()
index["start"], index["n_frames"] = start, n_frames
index.to_csv(OUT / "train_index.csv", index=False)
print("saved train_mels.npy + train_index.csv")

## Background-noise soundscapes
A random sample of the unlabelled 60 s soundscapes. They come from the same recorders as the
test set, so mixing them into training clips teaches the model what test-time noise sounds
like. (They may contain unlabelled calls; that is accepted for augmentation.)

In [ ]:
def noise_mel(fname):
    return to_mel_u8(read_audio(DATA / "train_soundscapes" / fname, CFG["noise_sec"] * CFG["sr"]))

scapes = sorted(p.name for p in (DATA / "train_soundscapes").glob("*.ogg"))
rng = np.random.default_rng(CFG["seed"])
n_noise = min(CFG["n_noise"], len(scapes)) if not LIMIT else min(5, len(scapes))
noise_files = sorted(rng.choice(scapes, n_noise, replace=False)) if n_noise else []
fpn = n_frames_for(CFG["noise_sec"] * CFG["sr"])
print(f"{len(scapes)} soundscapes available, using {n_noise} -> "
      f"{n_noise * fpn * CFG['n_mels'] / 1e9:.2f} GB")

if n_noise:
    N = np.lib.format.open_memmap(OUT / "noise_mels.npy", mode="w+", dtype=np.uint8,
                                  shape=(n_noise, fpn, CFG["n_mels"]))
    t0 = time.time()
    with mp.get_context("fork").Pool(os.cpu_count()) as pool:
        for i, mel in enumerate(pool.imap(noise_mel, noise_files, chunksize=8)):
            N[i] = mel
    N.flush(); del N
    pd.DataFrame({"filename": noise_files}).to_csv(OUT / "noise_index.csv", index=False)
    print(f"saved noise_mels.npy in {time.time() - t0:.0f}s")

## Save settings and sanity checks

In [ ]:
cfg_out = dict(CFG, n_clips=len(train), total_frames=total, n_noise=n_noise,
               frames_per_noise=fpn, frames_per_5s=n_frames_for(5 * CFG["sr"]),
               decode="dB re clip max = value / 255 * store_db - store_db; for a model input window: "
                      "db -= db.max(); x = (max(db, -top_db) + top_db) / top_db",
               layout="train_mels.npy is (frames, n_mels); transpose a slice to (n_mels, frames)")
json.dump(cfg_out, open(OUT / "mels_cfg.json", "w"), indent=1)

X = np.load(OUT / "train_mels.npy", mmap_mode="r")
idx = pd.read_csv(OUT / "train_index.csv")
assert idx["n_frames"].sum() == X.shape[0]

# Check: random 5 s windows (as the random-crop ablation will cut them) must match what the
# baseline / inference computes from that exact piece of audio.
SD, TD = CFG["store_db"], CFG["top_db"]
f5 = cfg_out["frames_per_5s"]
hop = CFG["hop_length"]
rng = np.random.default_rng(0)
errs, deep = [], 0
for j in rng.choice(len(idx), min(50, len(idx)), replace=False):
    r = idx.iloc[j]
    off = int(rng.integers(0, r.n_frames - f5 + 1))
    db = X[r.start + off : r.start + off + f5].T / 255.0 * SD - SD
    deep += db.max() < -(SD - TD)                 # crop >40 dB quieter than the clip peak
    stored = (np.maximum(db - db.max(), -TD) + TD) / TD
    y = read_audio(DATA / "train_audio" / r.filename, n_samples_for(sf.info(DATA / "train_audio" / r.filename).frames))
    seg = y[off * hop : off * hop + 5 * CFG["sr"]]
    seg = np.pad(seg, (0, 5 * CFG["sr"] - len(seg)))
    m = librosa.power_to_db(librosa.feature.melspectrogram(
            y=seg, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=hop,
            n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"]), ref=np.max, top_db=TD)
    base = (m + TD) / TD
    # Frames within ~n_fft/2 of the window edges see different neighbouring audio, so skip them.
    errs.append(np.abs(stored[:, 5:-5] - base[:, 5:-5]).ravel())
err = np.concatenate(errs)
print(f"check vs inference-style mel on {len(errs)} random crops: median abs diff {np.median(err):.4f}, "
      f"99th pct {np.percentile(err, 99):.4f}, crops >40 dB below clip peak: {deep}")
assert np.percentile(err, 99) < 0.05, "stored windows do not match inference-style mels"

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
for ax, (_, rr) in zip(axes, idx.sample(3, random_state=1).iterrows()):
    ax.imshow(X[rr.start : rr.start + rr.n_frames].T, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"{rr.primary_label}  ({rr.n_frames} frames ≈ {rr.n_frames * CFG['hop_length'] / CFG['sr']:.0f} s)")
plt.tight_layout(); plt.savefig(OUT / "example_whole_clip_mels.png", dpi=120); plt.show()

for f in sorted(OUT.iterdir()):
    print(f"{f.name:32s} {f.stat().st_size / 1e6:10.1f} MB")